# Stage 2 — Instruction Fine-Tuning (SFT)
Continues from Stage 1 adapter. Trains on Q&A pairs so the model learns to answer questions.

**Runtime → T4 GPU.**

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = "/content/drive/MyDrive/domain-ai-assistant-finetuning"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/reports", exist_ok=True)
print("Project dir:", PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/domain-ai-assistant-finetuning


## 2. Install dependencies

In [2]:
%%capture

# Install Unsloth and let it install compatible dependencies
!pip install -U --no-cache-dir "unsloth[colab-new]"

# Install only packages that Unsloth does not necessarily install
!pip install -U bitsandbytes


## 3. Load instruction dataset
Expects `data/instruction_dataset.jsonl` — the merged flashcard + patient-phrased file from the data build step.

In [3]:
import json
from datasets import Dataset

data_path = f"{PROJECT_DIR}/data/instruction_dataset.jsonl"
rows = []
with open(data_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} instruction examples")
assert len(rows) >= 100, f"Only {len(rows)} examples -- need >= 100"
dataset = Dataset.from_list(rows)
print(dataset[0])


Loaded 1271 instruction examples
{'instruction': 'What are the possible causes of low glucose and high C-peptide levels?', 'response': 'Low glucose and high C-peptide levels can be caused by an insulinoma or the use of sulfonylurea drugs.'}


## 4. Load Stage 1 checkpoint + apply LoRA
Loading the Stage 1 adapter restores its weights. We then call `get_peft_model` to make them trainable for Stage 2. This is the correct way to continue training an adapter — not a second separate adapter on top.

In [4]:
from unsloth import FastLanguageModel
import torch

STAGE1_PATH = f"{PROJECT_DIR}/checkpoints/stage1_non_instruction_final"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Make adapter weights trainable for Stage 2
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Stage 1 adapter loaded and made trainable for SFT.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


Stage 1 adapter loaded and made trainable for SFT.


## 5. Format prompts

In [5]:
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    text = (
        PROMPT_TEMPLATE.format(instruction=example["instruction"])
        + example["response"]
        + EOS_TOKEN
    )
    return {"text": text}

formatted_dataset = dataset.map(format_example)
print(formatted_dataset[0]["text"][:400])


Map:   0%|          | 0/1271 [00:00<?, ? examples/s]

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the possible causes of low glucose and high C-peptide levels?

### Response:
Low glucose and high C-peptide levels can be caused by an insulinoma or the use of sulfonylurea drugs.<|im_end|>


## 6. Train

In [6]:
from transformers import TrainingArguments
from trl import SFTTrainer
import torch
import traceback

CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints/stage2_sft"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_steps=10,
    max_steps=400,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    logging_steps=10,

    # Disable checkpoint saving
    save_strategy="no",

    optim="adamw_8bit",

    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

try:
    trainer.train()
    print("Training completed successfully!")

except Exception:
    traceback.print_exc()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1271 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,271 | Num Epochs = 3 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.805114
20,1.053537
30,0.875552
40,0.866287
50,0.833107
60,0.800761
70,0.868925
80,0.821610
90,0.800062
100,0.815260


Training completed successfully!


## 7. Save SFT model to Drive

In [7]:
SAVE_PATH = f"{PROJECT_DIR}/checkpoints/stage2_sft_final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved to:", SAVE_PATH)


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints/stage2_sft_final/tokenizer_config.json.


Saved to: /content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints/stage2_sft_final


## 8. Run 10 test questions — copy output into `reports/sft_model_comparison.md`

In [8]:
FastLanguageModel.for_inference(model)

TEST_QUESTIONS = ['How can I tell if my blood sugar is dangerously low?', 'Can I skip my metformin if I feel fine today?', "What's the difference between type 1 and type 2 diabetes?", 'Is it safe to exercise right after taking insulin?', 'Why am I so thirsty all the time?', 'Can type 2 diabetes be cured if I lose enough weight?', 'What should I do if my blood sugar reads 250?', 'Is it normal for blood sugar to vary day to day?', 'Can stress affect my blood sugar?', "What's considered a dangerously low blood sugar level?"]

for q in TEST_QUESTIONS:
    prompt = PROMPT_TEMPLATE.format(instruction=q)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    answer = tokenizer.decode(output[0], skip_special_tokens=True).split("### Response:")[-1].strip()
    print(f"Q: {q}\nA: {answer}\n{'-'*60}")


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Q: How can I tell if my blood sugar is dangerously low?
A: If your blood sugar stays below 60, it's time to call for help or check your pulse/fingertips.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Can I skip my metformin if I feel fine today?
A: You should always take your metformin as prescribed, even if you feel fine today. Metformin works by reducing glucose production in the liver and improving insulin sensitivity, which can help control blood sugar levels over time. Skipping doses or stopping the medication without guidance from your healthcare provider can lead to poor glycemic control and potentially serious health complications. If you miss a dose or have doubts about whether you've taken enough, talk to your doctor right away for instructions on what to do.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What should I do if my blood sugar reads 250?
A: If your blood sugar reads 250, you may need to take fast-acting insulin or regular (slow-acting) insulin. This will help lower your blood sugar quickly and prevent it from getting too high. It's important to monitor your blood sugar regularly and adjust your treatment plan as needed to keep it within a healthy range.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Is it normal for blood sugar to vary day to day?
A: Yes, it is normal for blood sugar levels to fluctuate throughout the day and even from one day to the next. Blood sugar levels are affected by many factors, including food intake, physical activity, stress, and medication use. Small fluctuations in blood sugar levels can be within a healthy range and do not necessarily indicate any underlying health problem. However, if you notice significant or persistent changes in your blood sugar levels, especially if they are below 70 mg/dL or above 140 mg/dL, talk to your healthcare provider to determine the cause and appropriate steps to manage your blood sugar levels.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Can stress affect my blood sugar?
A: Yes, stress can cause fluctuations in your blood sugar levels. When you're stressed, your body releases cortisol and adrenaline, which can raise your blood glucose levels. This is because cortisol stimulates the liver to release stored glucose into the bloodstream. However, if these stress responses are frequent or intense, they can lead to long-term health problems such as insulin resistance and type 2 diabetes. If you notice changes in your blood sugar levels, it's important to talk to your doctor to determine the underlying causes and appropriate treatment.
------------------------------------------------------------
Q: What's considered a dangerously low blood sugar level?
A: A blood sugar reading below 60 mg/dL is generally considered dangerous, as it can lead to symptoms such as confusion and seizures.
------------------------------------------------------------


In [9]:
!ls -lh /content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints

total 16K
drwx------ 2 root root 4.0K Jul  7 07:01 stage1_non_instruction
drwx------ 2 root root 4.0K Jul  9 07:50 stage1_non_instruction_final
drwx------ 3 root root 4.0K Jul  9 08:06 stage2_sft
drwx------ 2 root root 4.0K Jul  9 08:30 stage2_sft_final


In [10]:
!ls -lh /content/drive/MyDrive/domain-ai-assistant-finetuning/data

total 752K
-rw------- 1 root root 714K Jul  7 06:23 instruction_dataset.jsonl
-rw------- 1 root root  29K Jul  7 06:20 non_instruction_data.txt
-rw------- 1 root root 7.3K Jul  4 07:07 patient_phrased_examples.jsonl
-rw------- 1 root root 2.0K Jul  4 07:07 README_dataset_build.md


In [11]:
!ls /content/drive/MyDrive/domain-ai-assistant-finetuning/scripts

build_preference_dataset.py  merge_instruction_dataset.py  test_base_model.py
filter_medalpaca.py	     scrape_medlineplus.py


In [12]:
%cd /content/drive/MyDrive/domain-ai-assistant-finetuning
!python scripts/build_preference_dataset.py

/content/drive/MyDrive/domain-ai-assistant-finetuning
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will 

In [13]:
!ls -lh /content/drive/MyDrive/domain-ai-assistant-finetuning/data

total 790K
-rw------- 1 root root 714K Jul  7 06:23 instruction_dataset.jsonl
-rw------- 1 root root  29K Jul  7 06:20 non_instruction_data.txt
-rw------- 1 root root 7.3K Jul  4 07:07 patient_phrased_examples.jsonl
-rw------- 1 root root  38K Jul  9 08:41 preference_dataset.jsonl
-rw------- 1 root root 2.0K Jul  4 07:07 README_dataset_build.md
